In [ ]:
from src.paths import dataset_dir, dataset_file, dataset_root, repository_root
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import scipy

import openai
import ast
client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])

from src.utils import *
import src.prompt as prompt
from src.data_loader import load_spatial_data_anndata

# data

In [ ]:
representetive_gene_list = ["Aqp4", "Hpcal1", "Pvalb", "Frem3", "Pcp4", "Krt17","Mobp", 
                          "Lamp5", "Rorb", "Fezf2", "Syt6", "Fa2h",
                          "Plp1", "Foxj1", "Gfap", "Cpne5", "Kcnip2",
                          "Bgn", "Cux2", "Etv1",
                          "Zmat4", "Rab3c"]
representetive_gene_list = [gene.upper() for gene in representetive_gene_list]

In [ ]:
config = load_config("configs/config_zeroshot_libd.yaml")
config.data_name = "151509"
config.refresh_paths()


In [ ]:
# --- Load data ---
data_path = str(dataset_dir("visium_libd", config.data_name))

adata = load_spatial_data_anndata(
    data_path=data_path,
    adata_file="filtered_feature_bc_matrix.h5",
    config=config,
    celltype_path=os.path.join("examples/visium_libd/deconv_result", f"celltype_proportions_{config.data_name}.csv"),
    optional_files=["metadata.tsv"]
)

In [ ]:
# Prepare neighbor data using the new function
neighbor_normalized_df, neighbor_normalized_df_genes, adj_matrix = prepare_neighbor_data(
    adata, config, representetive_gene_list
)

# prompt

In [ ]:
unique_layers = adata.obs[config.name_truth].unique()
domain_mapping = {i: layer for i, layer in enumerate(unique_layers)}
config.domain_mapping = domain_mapping

# cell_names_mapping ={'Astro': 'Astrocyte',
#  'EndoMural': 'Endothelial and mural cells',
#  'Excit_L2_3': 'Excitatory neuron layer 2/3',
#  'Excit_L3': 'Excitatory neuron layer 3',
#  'Excit_L3_4_5': 'Excitatory neuron layer 3/4/5',
#  'Excit_L4': 'Excitatory neuron layer 4',
#  'Excit_L5': 'Excitatory neuron layer 5',
#  'Excit_L5_6': 'Excitatory neuron layer 5/6',
#  'Excit_L6': 'Excitatory neuron layer 6',
#  'Inhib': 'Inhibitory neuron',
#  'Micro': 'Microglia',
#  'OPC': 'Oligodendrocyte precursor cell',
#  'Oligo': 'Oligodendrocyte'}

# config.cell_names_mapping = cell_names_mapping







In [ ]:
config.domain_mapping.values()

In [ ]:

# IMPORTANT: check input and prompt_func
if config.Graph_type == "countPlusGenes":
    input_df = neighbor_normalized_df
    df_extra = neighbor_normalized_df_genes
    prompt_func = prompt.zeroshot_celltype_geneorder
elif config.Graph_type == "count":
    input_df = neighbor_normalized_df
    df_extra = None
    prompt_func = prompt.zeroshot_celltype
elif config.Graph_type == "GeneOnly":
    input_df = neighbor_normalized_df_genes
    df_extra = None
    prompt_func = prompt.zeroshot_geneorder
else:
    raise ValueError(f"Graph_type {config.Graph_type} not supported")


In [ ]:
i = np.where(adata.obs.loc[:,config.name_truth].values == "WM")[0][4]
print(prompt_func(input_df, rows=[i], config=config))

print(i)

In [ ]:
id_for_cellchat = [0,2,19,14,21,17,28,29,38,61,1,8,9,15,16,25,44,50,70,71,7,13,18,30,4,5,11,12,23,3,6,20,63,69]

# GPT

In [ ]:
print(config.data_name)
print(config.gpt_model)
print(config.replicate)

In [ ]:
# choose correct data and prompt
generate_json_end2end(input_df, 
                      config, 
                      prompt_func, 
                      batch_size = 2000,
                      max_completion_tokens = 512,  # key to control the cost, expecially for o3-mini
                      n_rows = 1,
                      df_extra = df_extra)

In [ ]:
# submit_end2end.py

import subprocess
cmd = f"nohup python -u -m src.submit_end2end configs/config_zeroshot_libd.yaml {config.data_name} {config.replicate} parallel > outs/zeroshot_{config.data_name}{config.replicate}.out 2>&1 &"
subprocess.run(cmd, check=True, text=True, shell=True)



In [ ]:
# retrive the results of the parallel runs
tag = "20250822_11"
cmd = f"nohup python -u -m src.retrive_batch_results_parallel configs/config_zeroshot_libd.yaml {config.data_name} {config.replicate} {tag} &"
subprocess.run(cmd, check=True, text=True, shell=True)


In [ ]:
# 初始化空列表以保存custom_id和content
gpt_results_df = pd.DataFrame()
n_batch = 3
for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']

                # extract outputs - handle both JSON format and text format
                extract_dict = extract_json_microenvironment(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=[custom_id])], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")
gpt_results_df.columns = ['zeroshot_gpt4o_mini']


In [ ]:
gpt_results_df

In [ ]:
gpt_results_df.zeroshot_gpt4o_mini.value_counts()

In [ ]:
gpt_results_df.index.difference(adata.obs.index)

In [ ]:
# # Replace strings in the first column of gpt_results_df that contain keywords plus '**' or '.'
# import re

# # Get the list of keywords from domain_mapping
# keywords = list(domain_mapping.values())

# # Create a regex pattern to capture any keyword possibly surrounded by other text
# pattern = r'.*(' + '|'.join(map(re.escape, keywords)) + r')[\*\.\s]*.*'

# # Replace the entire string with the captured keyword only if it could be not unknown
# for nichtype in gpt_results_df.zeroshot_gpt4o_mini.value_counts()[gpt_results_df.zeroshot_gpt4o_mini.value_counts()<3].index:
#     gpt_results_df.loc[gpt_results_df.zeroshot_gpt4o_mini == nichtype, "zeroshot_gpt4o_mini"] = gpt_results_df.loc[gpt_results_df.zeroshot_gpt4o_mini == nichtype, "zeroshot_gpt4o_mini"].str.replace(pattern, r'\1', regex=True)


In [ ]:
# if the number of the cell type is less than 4, set it to unknown
for nichtype in gpt_results_df.zeroshot_gpt4o_mini.value_counts()[gpt_results_df.zeroshot_gpt4o_mini.value_counts()<4].index:
    gpt_results_df.loc[gpt_results_df.zeroshot_gpt4o_mini == nichtype, "zeroshot_gpt4o_mini"] = "unknown"


# Gemini

In [ ]:
import google.generativeai as genai
import pickle
import time



genai.configure(api_key=os.environ["API_KEY"])
model = genai.GenerativeModel("gemini-1.5-pro")
gen_config=genai.types.GenerationConfig(temperature=1.0, max_output_tokens=1000)

In [ ]:
gemini_results_df, store_responses = run_gemini(model, gen_config, 
                                                neighbor_normalized_df, config, 
                                                prompt.zeroshot_celltype_geneorder, n_rows=1, 
                                                df_extra = neighbor_normalized_df_genes,
                                                column_name="zeroshot_gemini")

with open(f'./gemini_results/{config.data_name}_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.pkl', 'wb') as file:
    pickle.dump(store_responses, file)

gemini_results_df.to_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")


In [ ]:
gemini_results_df = pd.read_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv", index_col=0)

In [ ]:
gemini_results_df.index.difference(neighbor_normalized_df.index)

In [ ]:
gemini_results_df.zeroshot_gemini.value_counts()

# plot and save

In [ ]:
# adata.obs.drop(columns=['zeroshot_gemini'], inplace=True)
# adata.obs.drop(columns=['zeroshot_gpt4o_mini'], inplace=True)

In [ ]:
# adata.obs = pd.read_csv(f"./gpt4omini_results/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")

In [ ]:
# sc.pl.spatial(adata, color=['zeroshot_gpt4o_mini_refined',  name_truth], library_id=config.data_name, size=1.4)
# print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['zeroshot_gpt4o_mini_refined']))
# print(normalized_mutual_info_score(adata.obs[config.name_truth], adata.obs['zeroshot_gpt4o_mini_refined']))

In [ ]:
# adata.obs = adata.obs.join(gemini_results_df)
adata.obs = adata.obs.join(gpt_results_df)
# adata.obs['zeroshot_gemini'] = adata.obs['zeroshot_gemini'].fillna("unknown")
adata.obs['zeroshot_gpt4o_mini'] = adata.obs['zeroshot_gpt4o_mini'].fillna("unknown")


In [ ]:
sc.pl.spatial(adata, color=['zeroshot_gpt4o_mini',  config.name_truth], library_id=config.data_name, size=1.4)
print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['zeroshot_gpt4o_mini']))
print(normalized_mutual_info_score(adata.obs[config.name_truth], adata.obs['zeroshot_gpt4o_mini']))

In [ ]:
# gemini_results_df.to_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")
# gpt_results_df.to_csv(f"./gpt4omini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")

# refine the niche

In [ ]:
adata.obs['zeroshot_gpt4o_mini_refined'] = relabel_cells(adj_matrix.toarray(), adata.obs['zeroshot_gpt4o_mini'])
print(normalized_mutual_info_score(adata.obs[config.name_truth], adata.obs['zeroshot_gpt4o_mini_refined']))

adata.obs.to_csv(f"./gpt4omini_results/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")

In [ ]:
print((f"./gpt4omini_results/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv"))

In [ ]:
metadata = pd.read_csv("examples/results/zeroshot_visium/refined/gemini_results/151507_with_refined_zeroshot_end2end_False_False_True_countPlusGenes_False_False_False_True_rep1R600.csv", index_col=0)

In [ ]:
metadata

In [ ]:
cellchat_results = pd.read_csv("examples/results/cellchat_results_151507.csv", index_col=0)




In [ ]:
normalized_mutual_info_score(cellchat_results['layer_guess'], cellchat_results['cellchat'])

In [ ]:
normalized_mutual_info_score(cellchat_results['layer_guess'], cellchat_results['gpt4omini'])

In [ ]:
normalized_mutual_info_score(cellchat_results['layer_guess'], cellchat_results['gemini'])

In [ ]:
normalized_mutual_info_score(cellchat_results['layer_guess'], cellchat_results['test finetune'])

In [ ]:
# remove nan
no_nan_cellchat_results = cellchat_results[cellchat_results['val finetune'].notna()]

In [ ]:
normalized_mutual_info_score(no_nan_cellchat_results['layer_guess'], no_nan_cellchat_results['val finetune'])

In [ ]:
cellchat_results.to_csv("examples/results/cellchat_results_151507.csv")